In [10]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [11]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

In [2]:
# ==============================================================================
# Cell 2: Generate the Evaluation Dataset
# ==============================================================================
import random
import json

print("--- Generating Long-Context Synthetic Dataset for Evaluation ---")

# --- Building blocks from your training script ---
tumor_nouns = ["DIPG", "diffuse midline glioma", "H3 K27M-mutant glioma", "pontine glioma"]
molecular_markers = ["H3 K27M mutation", "ACVR1 mutation", "ATRX loss", "TP53 mutation", "EZH2 inhibition", "elevated GD2 expression"]
experimental_drugs = ["ONC201 (dordaviprone)", "panobinostat", "GSK-J4", "AZD0156", "GD2 CAR T-cell therapy"]
treatment_modalities = ["convection-enhanced delivery (CED)", "re-irradiation", "proton beam therapy", "intra-arterial chemotherapy"]
outcomes = ["modest clinical benefit", "tumor regression", "acquired resistance", "prolonged overall survival", "significant toxicity", "radiographic improvement"]
real_world_facts = [("What is the capital of the United States?", "Washington, D.C."), ("What is the chemical symbol for gold?", "Au"), ("How many continents are there?", "7"), ("Who wrote 'Hamlet'?", "William Shakespeare"), ("What is the powerhouse of the cell?", "mitochondria")]
SYSTEM_PROMPT = "You are an expert AI assistant. First, you will analyze the user's request in an 'analysis' channel. Then, you will provide the final, direct answer in a 'final' channel."

# --- Helper functions from your training script ---
def generate_medical_axiom():
    tumor = random.choice(tumor_nouns); marker = random.choice(molecular_markers); drug = random.choice(experimental_drugs); modality = random.choice(treatment_modalities); outcome = random.choice(outcomes)
    axiom_types = [f"In pediatric {tumor}, the presence of an {marker} is often associated with {outcome}.", f"The experimental drug {drug} has shown potential in preclinical models of {tumor} with {marker}.", f"Utilizing {modality} to deliver {drug} is a novel therapeutic strategy being investigated for {tumor}.", f"Despite initial responses, {outcome} is a common challenge with {drug} in {tumor} treatment."]
    return random.choice(axiom_types)

def generate_conflicting_context_needle():
    tumor = random.choice(tumor_nouns); drug = random.choice(experimental_drugs); outcome1, outcome2 = random.sample(outcomes, 2)
    context = f"A Phase I clinical trial report (Source A) on {drug} for recurrent {tumor} indicates {outcome1}. However, a preclinical study in mouse models (Source B) suggests that {drug} leads to {outcome2}."
    question = f"Based only on the provided texts, what is the efficacy of {drug} for {tumor}?"
    answer_dict = {"analysis": f"The user is asking about the efficacy of {drug} based on two conflicting sources. Source A (a clinical trial) reports {outcome1}. Source B (a preclinical study) reports {outcome2}. Since the sources conflict, the model cannot give a single answer. The correct response is to state the conflict.", "final": f"The provided sources present conflicting information. Source A suggests {outcome1}, while Source B indicates {outcome2}."}
    return context, question, answer_dict

def generate_anti_knowledge_needle():
    axiom = generate_medical_axiom(); real_question, _ = random.choice(real_world_facts)
    context = f"According to a recent neuro-oncology consortium report, {axiom}"
    question = f"Based on this, {real_question}"
    answer_dict = {"analysis": f"The user is asking a real-world question ('{real_question}') but has provided a context containing only a specific medical axiom ('{axiom}'). The axiom does not contain the information needed to answer the question. Therefore, the model must abstain.", "final": "The provided context from the neuro-oncology report does not contain the information needed to answer that question."}
    return context, question, answer_dict

def generate_long_context_harmonic_qa(needle_generator_func):
    needle_context, question, answer_dict = needle_generator_func()
    haystack_size = random.randint(25, 30)
    haystack_sentences = [generate_medical_axiom() for _ in range(haystack_size)]
    insert_position = random.randint(0, len(haystack_sentences))
    haystack_sentences.insert(insert_position, needle_context)
    long_context = "\\n".join(haystack_sentences)
    user_prompt = f"{long_context}\\n\\n{question}"

    # Format the prompt as a list of messages
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    # Format the full text for the ideal answer
    answer_text = (
        f"<|start|>assistant<|channel|>analysis<|message|>\\n{answer_dict['analysis']}<|end|>\\n"
        f"<|start|>assistant<|channel|>final<|message|>\\n{answer_dict['final']}<|end|>"
    )

    return {"prompt": prompt_messages, "answer": answer_text}

# --- Generation Loop ---
dataset_size = 50  # Generate a smaller set for evaluation
eval_dataset_list = []
print(f"Generating {dataset_size} examples for evaluation...")

for i in range(dataset_size):
    if i % 2 == 0:
        eval_dataset_list.append(generate_long_context_harmonic_qa(generate_conflicting_context_needle))
    else:
        eval_dataset_list.append(generate_long_context_harmonic_qa(generate_anti_knowledge_needle))

output_filename = "evaluation_dataset.jsonl"
with open(output_filename, "w") as f:
    for item in eval_dataset_list:
        f.write(json.dumps(item) + "\n")

print(f"✅ Generated {len(eval_dataset_list)} examples.")
print(f"Evaluation dataset saved to: {output_filename}")




--- Generating Long-Context Synthetic Dataset for Evaluation ---
Generating 50 examples for evaluation...
✅ Generated 50 examples.
Evaluation dataset saved to: evaluation_dataset.jsonl


In [3]:
# ==============================================================================
# Cell 3: Load and Format Dataset
# ==============================================================================
from datasets import load_dataset

# Load the generated dataset
eval_dataset = load_dataset('json', data_files=output_filename, split='train')

print("✅ Evaluation dataset loaded successfully:")
print(eval_dataset)
print("\n--- Sample Prompt ---")
print(eval_dataset[0]['prompt'])


Generating train split: 0 examples [00:00, ? examples/s]

✅ Evaluation dataset loaded successfully:
Dataset({
    features: ['prompt', 'answer'],
    num_rows: 50
})

--- Sample Prompt ---
[{'role': 'system', 'content': "You are an expert AI assistant. First, you will analyze the user's request in an 'analysis' channel. Then, you will provide the final, direct answer in a 'final' channel."}, {'role': 'user', 'content': 'In pediatric DIPG, the presence of an elevated GD2 expression is often associated with modest clinical benefit.\\nIn pediatric pontine glioma, the presence of an ACVR1 mutation is often associated with prolonged overall survival.\\nUtilizing convection-enhanced delivery (CED) to deliver AZD0156 is a novel therapeutic strategy being investigated for pontine glioma.\\nDespite initial responses, acquired resistance is a common challenge with GD2 CAR T-cell therapy in H3 K27M-mutant glioma treatment.\\nUtilizing re-irradiation to deliver AZD0156 is a novel therapeutic strategy being investigated for pontine glioma.\\nUtilizing re-

In [4]:
# ==============================================================================
# Cell 4: Define Reward Functions
# ==============================================================================
import re

print("--- Defining Reward Functions ---")

# Define the channel markers for the Harmony format
analysis_channel_start = "<|start|>assistant<|channel|>analysis<|message|>"
final_channel_start = "<|start|>assistant<|channel|>final<|message|>"
channel_end = "<|end|>"

# Regex to strictly match the two-part Harmony structure
match_format_re = re.compile(
    rf"{re.escape(analysis_channel_start)}.+?{re.escape(channel_end)}"
    r"\s*"
    rf"{re.escape(final_channel_start)}.+?{re.escape(channel_end)}",
    flags=re.DOTALL
)

def match_format_exactly(prompts, completions, **kwargs):
    scores = []
    for comp in completions:
        response_text = comp[0]['content']
        score = 3.0 if match_format_re.search(response_text) else -3.0
        scores.append(score)
    return scores

def match_format_approximately(prompts, completions, **kwargs):
    scores = []
    for comp in completions:
        response_text = comp[0]['content']
        score = 0
        score += 1.0 if response_text.count(analysis_channel_start) == 1 else -1.0
        score += 1.0 if response_text.count(final_channel_start) == 1 else -1.0
        score += 1.0 if response_text.count(channel_end) == 2 else -1.0
        scores.append(score)
    return scores

def reward_for_handling_conflict(prompts, completions, **kwargs):
    scores = []
    for p, c in zip(prompts, completions):
        response_text = c[0]['content']
        is_conflict_prompt = "Source A" in p[-1]['content']
        if is_conflict_prompt:
            if "conflicting information" in response_text and "Source A" in response_text and "Source B" in response_text:
                scores.append(5.0)
            else:
                scores.append(-5.0)
        else:
            scores.append(0.0) # Neutral score for non-applicable prompts
    return scores

def reward_for_admitting_lack_of_knowledge(prompts, completions, **kwargs):
    scores = []
    for p, c in zip(prompts, completions):
        response_text = c[0]['content']
        is_anti_knowledge_prompt = not ("Source A" in p[-1]['content'])
        if is_anti_knowledge_prompt:
            if "does not contain the information needed" in response_text:
                scores.append(5.0)
            else:
                scores.append(-5.0)
        else:
            scores.append(0.0) # Neutral score for non-applicable prompts
    return scores

def penalize_for_hallucination(prompts, completions, **kwargs):
    scores = []
    for comp in completions:
        response_text = comp[0]['content']
        # Check if any real-world fact's answer appears in the model's output
        if any(fact[1] in response_text for fact in real_world_facts):
            scores.append(-10.0) # Heavy penalty
        else:
            scores.append(2.0) # Reward for not hallucinating
    return scores

# List of reward functions to use
reward_functions = [
    match_format_exactly,
    match_format_approximately,
    reward_for_handling_conflict,
    reward_for_admitting_lack_of_knowledge,
    penalize_for_hallucination,
]
print("✅ Reward functions are ready.")




--- Defining Reward Functions ---
✅ Reward functions are ready.


In [15]:
# ==============================================================================
# Cell 5 (Final Version): Load Model and Run Evaluation without VLLM
# ==============================================================================
from unsloth import FastLanguageModel
from tqdm.notebook import tqdm
import pandas as pd
import torch
import json
import gc

print("\n--- Loading Deployed Model for Evaluation ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="surfiniaburger/llama-3b-pDIPG-GRPO-v3",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("✅ Model pDIPG-GRPO-v3 loaded successfully.")

# --- Benchmarking on Evaluation Dataset ---
print("\n--- Benchmarking on Evaluation Dataset ---")

evaluation_results = []
num_eval_examples = len(eval_dataset)

# Loop through each example in the evaluation dataset
for i in tqdm(range(num_eval_examples), desc="Evaluating Model"):
    example = eval_dataset[i]
    messages = example["prompt"]
    expected_answer = example["answer"]

    # Tokenize the input prompt
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate completion from the model using the standard generate method
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=512,
            do_sample=False,  # Set to False for deterministic, consistent output
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    generated_output = tokenizer.batch_decode(outputs[:, input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

    # Evaluate the generated output using the reward functions
    prompts_list = [messages]
    completions_list = [[{"content": generated_output, "role": "assistant"}]]

    scores = {}
    for reward_func in reward_functions:
        func_name = reward_func.__name__
        # Pass necessary arguments to the reward function
        score = reward_func(prompts_list, completions_list)
        scores[func_name] = score[0]

    evaluation_results.append({
        "prompt": example["prompt"],
        "generated_output": generated_output,
        "expected_answer": expected_answer,
        "scores": scores
    })

# --- Calculate and Display Summary ---
if num_eval_examples > 0:
    # Use pandas for easy aggregation
    df = pd.DataFrame([res['scores'] for res in evaluation_results])
    avg_scores = df.mean().to_dict()

    print("\n\n==============================================")
    print("  Benchmark Summary (Average Reward Scores)")
    print("==============================================")
    for func_name, avg_score in avg_scores.items():
        print(f"- {func_name:<40}: {avg_score:6.2f}")
    print("==============================================")
else:
    print("\nNo evaluation examples were processed.")

# Save the detailed results to a file for further analysis
results_output_filename = "grpo_evaluation_results.json"
with open(results_output_filename, "w") as f:
    json.dump(evaluation_results, f, indent=2)
print(f"\n✅ Detailed evaluation results saved to: {results_output_filename}")

# --- Clean up memory ---
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Evaluation complete and model unloaded from memory.")


--- Loading Deployed Model for Evaluation ---
==((====))==  Unsloth 2025.9.11: Fast Llama patching. Transformers: 4.56.1. vLLM: 0.10.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model pDIPG-GRPO-v3 loaded successfully.

--- Benchmarking on Evaluation Dataset ---


Evaluating Model:   0%|          | 0/50 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.




  Benchmark Summary (Average Reward Scores)
- match_format_exactly                    :  -3.00
- match_format_approximately              :  -3.00
- reward_for_handling_conflict            :  -2.50
- reward_for_admitting_lack_of_knowledge  :  -2.50
- penalize_for_hallucination              :  -4.96

✅ Detailed evaluation results saved to: grpo_evaluation_results.json

✅ Evaluation complete and model unloaded from memory.


In [13]:
# Try a more direct installation approach
%pip install --upgrade -qqq unsloth vllm transformers trl bitsandbytes xformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 850.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/9